# EXP-20260828-graph-01

```text
실험 ID: EXP-20260828-graph-01
생성 기준일: 2026-08-28
기준 소스: src 최신본
실험 목적: Graph 검색 가능 범위 확인 (로컬 pyoxigraph, cutoff 2026-08-24)
```

이 노트북은 생성 당시 `src/` 의 복사본을 가진 독립 실험 공간이다 (계획 v4 §3).
`%%module` 셀 수정은 `src/` 에 자동 반영되지 않으며, `sync_to_py(dry_run=False)` 는 최종 채택 시에만 실행한다.

**전제**: `artifacts/oxigraph` 가 cutoff 2026-08-24 로 재빌드돼 있어야 한다
(기존 07-11 cutoff 빌드는 08-24 릴리스 관계 573노드를 제외하고 있었다 — 백업: `artifacts/oxigraph.pre_0828_cutoff0711`).

In [4]:
# === 셀 매직 정의: 각 셀을 실제 모듈로 등록한다 ===
# 사용법: 셀 첫 줄에 `%%module <모듈명> <src 기준 경로>`.
# 셀을 수정하고 재실행하면 sys.modules 가 교체되므로,
# 그 모듈을 import 하는 하위 셀들을 다시 실행하면 수정본이 반영된다.
import json as _json
import sys as _sys
import types as _types
from pathlib import Path

from IPython.core.magic import register_cell_magic

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
NB_PATH = REPO_ROOT / "test" / "notebook" / "experiments" / "agent_graph_0828.ipynb"
RESULTS_DIR = REPO_ROOT / "test" / "notebook" / "experiments" / "results"
RESULTS_DIR.mkdir(exist_ok=True)


@register_cell_magic("module")
def _module_magic(line, cell):
    name, relpath = line.split()
    mod = _types.ModuleType(name)
    mod.__file__ = str(REPO_ROOT / "src" / relpath)
    _sys.modules[name] = mod
    parts = name.split(".")
    for i in range(1, len(parts)):
        pkg = ".".join(parts[:i])
        parent = _sys.modules.setdefault(pkg, _types.ModuleType(pkg))
        setattr(parent, parts[i], _sys.modules.get(name) if i == len(parts) - 1 else _sys.modules.setdefault(".".join(parts[:i + 1]), _types.ModuleType(".".join(parts[:i + 1]))))
    exec(compile(cell, mod.__file__, "exec"), mod.__dict__)
    print(f"registered: {name}")


def sync_to_py(dry_run=True):
    """%%module 셀을 src/*.py 로 되쓴다. 최종 채택 시에만 사용 (계획 v4 §10)."""
    nb = _json.loads(NB_PATH.read_text(encoding="utf-8"))
    for c in nb["cells"]:
        src = "".join(c["source"])
        if c["cell_type"] != "code" or not src.startswith("%%module "):
            continue
        first, _, body = src.partition("\n")
        _, name, relpath = first.split()
        target = REPO_ROOT / "src" / relpath
        old = target.read_text(encoding="utf-8") if target.exists() else None
        if old == body:
            print(f"  same: {relpath}")
        elif dry_run:
            print(f"CHANGED: {relpath}  (dry_run — 반영하려면 sync_to_py(dry_run=False))")
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_text(body, encoding="utf-8")
            print(f"WROTE: {relpath}")


In [5]:
import json

man_path = REPO_ROOT / "artifacts" / "oxigraph" / "manifest.json"
man = json.loads(man_path.read_text(encoding="utf-8"))
assert man["cutoff"] == "2026-08-24", (
    f"store cutoff={man['cutoff']} — 2026-08-24 재빌드 필요 "
    "(kb.build_graph 의 CUTOFF 를 08-24 로 바꿔 build(); 기존 store 는 백업 후)")
print("store:", f"{man['triple_count']:,} triples, cutoff", man["cutoff"],
      "| excluded_future:", man["excluded_future_nodes"])

store: 1,628,311 triples, cutoff 2026-08-24 | excluded_future: 0


## `src/config.py` (verbatim)

In [6]:
%%module config config.py
# -*- coding: utf-8 -*-
"""경로·모델 상수. 채권 MVP 범위."""
from pathlib import Path

# config.py 는 src/ 안에 있다. .parent = src/, 한 번 더 올려야 저장소 루트다.
# 한 번만 올리면 ARTIFACTS 가 src/artifacts 를 새로 만들며 조용히 캐시를 잃는다.
ROOT = Path(__file__).resolve().parent.parent

# 지시서는 ontology/bond.ttl 하나를 가정하지만 이 저장소의 스키마는 common + 도메인 4로 갈려 있다.
# 테스트 질문의 '위험등급'(fp:RiskGrade·fp:riskGradeLevel)은 common.ttl에만 있어서
# bond_kr.ttl만 인덱싱하면 4문항 중 1문항을 못 답한다. 둘 다 넣는다.
BOND_TTL_PATHS = [ROOT / "ontology" / "bond_kr.ttl", ROOT / "ontology" / "common.ttl"]

ARTIFACTS = ROOT / "artifacts"

# 스키마 벡터 인덱스는 PostgreSQL + pgvector 에 둔다(FAISS 에서 이전, 2026-08-22).
# 이전 근거는 vectordb_test/results/1_pgvector_test_report.md — cosine 점수가
# FAISS(정규화 후 IndexFlatIP)와 최대 오차 5.03e-07 로 일치해 임계값을 그대로 쓴다.
# 비밀번호를 포함하므로 DSN 을 로그에 찍지 않는다. 접속 정보는 환경변수로 덮을 수 있다.
import os

# Azure Data API
# 현재 공개 테스트 API는 임시 주소다. 운영에서는 환경변수로 반드시 덮어쓴다.
FINANCIAL_DATA_API_URL = os.environ.get(
    "FINANCIAL_DATA_API_URL",
    "http://40.82.145.44:8000",
)
FINANCIAL_DATA_RELEASE_ID = os.environ.get(
    "FINANCIAL_DATA_RELEASE_ID",
    "financial-products-2026-08-24@"
    "ddb3d994a4a5115a75bed7efa9c4cd0f6655f95b0a49f3b0e3c01b2bf8301a38",
)
DATA_API_TIMEOUT_SECONDS = float(
    os.environ.get("DATA_API_TIMEOUT_SECONDS", "10")
)

# Direct PostgreSQL connection.
# Existing rdb/bond_schema tools still use this configuration.
BOND_DB = {
    "host": os.environ.get("PGHOST", "127.0.0.1"),
    "port": os.environ.get("PGPORT", "5432"),
    "user": os.environ.get("PGUSER", "postgres"),
    "password": os.environ.get("PGPASSWORD", "postgres"),
    "dbname": os.environ.get("PGDATABASE", "mafest"),
}
# 유닉스 소켓은 peer 인증에 걸린다. host 를 명시해 TCP 로 붙는다.
BOND_DSN = " ".join(f"{k}={v}" for k, v in BOND_DB.items())
BOND_TABLE = "bond_schema_terms"
EMBED_DIM = 1024

BOND_TOP_K = 5
# cosine 점수 하한. 실측상 0.36~0.38대는 무관한 용어(자회사 관계 등)가 섞인다.
# 빈약한 근거를 주면 모델이 일반 지식으로 메워 근거 없는 단정이 나온다.
BOND_SCORE_FLOOR = 0.45

EMBEDDING_MODEL = "bge-m3"        # 1024차원, cosine (CLOVA Studio)
# 1단계 Query Frame 추출. HCX-005·HCX-DASH-002 와 대표 4문항으로 비교해 정했다 —
# 스키마 준수 4/4 vs 2/4 vs 1/4. DASH-002 의 속도 이점은 프롬프트가 길어지면서
# 사라졌다(출력 토큰이 지연을 지배한다). 근거: vectordb_test/4_query_frame_v1/4_result_query_frame_v1.md
FRAME_MODEL = "HCX-007"
ANSWER_MODEL = "HCX-005"          # 답변 생성
CHAT_TIMEOUT_SECONDS = 13           # API tail stall은 재시도 없이 ABSTAIN해 15초 E2E를 지킨다

CLOVA_HOST = "https://clovastudio.stream.ntruss.com"


registered: config


## `tools.graph` = src verbatim + 실험 확장(템플릿 함수)

In [7]:
%%module tools.graph tools/graph.py
# -*- coding: utf-8 -*-
"""읽기 전용 pyoxigraph SPARQL과 최초 관계 vertical slice."""
from __future__ import annotations

import re
from functools import lru_cache

from config import ARTIFACTS

try:
    from pyoxigraph import Store
except ImportError as exc:  # pragma: no cover - 설치 안내 경로
    raise SystemExit("pyoxigraph 미설치 — python3 -m pip install -r requirements.txt") from exc

STORE_PATH = ARTIFACTS / "oxigraph"
MAX_ROWS = 10_000
_FORBIDDEN = re.compile(
    r"\b(?:ADD|CLEAR|COPY|CREATE|DELETE|DROP|INSERT|LOAD|MOVE|SERVICE|WITH)\b",
    re.IGNORECASE,
)


@lru_cache(maxsize=1)
def _store() -> Store:
    if not STORE_PATH.is_dir():
        raise RuntimeError("Graph store 미구축 — python3 src/kb/build_graph.py")
    return Store.read_only(str(STORE_PATH))


def _value(term):
    if term is None:
        return None
    return term.value


def sparql(query: str) -> bool | list[dict]:
    """SELECT/ASK만 허용한다. Agent는 아래 고정 template 함수만 호출한다."""
    text = query.lstrip()
    text = re.sub(r"(?is)^(?:PREFIX\s+\w*:\s*<[^>]+>\s*)+", "", text).lstrip()
    kind = text.split(None, 1)[0].upper() if text else ""
    if kind not in {"SELECT", "ASK"} or _FORBIDDEN.search(query):
        raise ValueError("Graph query는 SERVICE 없는 SELECT/ASK만 허용합니다")
    result = _store().query(query)
    if kind == "ASK":
        return bool(result)
    variables = [v.value for v in result.variables]
    rows = []
    for solution in result:
        if len(rows) >= MAX_ROWS:
            raise ValueError(f"Graph 결과가 상한 {MAX_ROWS:,}행을 초과했습니다")
        rows.append({name: _value(solution[name]) for name in variables})
    return rows


ECOPRO_HOLDING_QUERY = """
PREFIX fp: <http://mafest.ai/product#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT DISTINCT ?etf ?etf_name ?child ?child_name ?security ?weight ?holding_as_of
                ?holding_source ?relation_as_of ?relation_source
                ?holding_document_title ?holding_document_publisher
                ?holding_document_date ?holding_document_quote
                ?relation_document_title ?relation_document_publisher
                ?relation_document_date ?relation_document_quote WHERE {
  ?parent a fp:Company ; rdfs:label "에코프로" ; fp:hasSubsidiary ?relation .
  ?relation fp:subsidiaryCompany ?child ; fp:asOf ?relation_as_of ;
            fp:sourceId ?relation_source ; fp:supportedBy ?relation_document .
  ?relation_document a fp:Document ; fp:documentTitle ?relation_document_title ;
            fp:documentPublisher ?relation_document_publisher ;
            fp:documentPublishedDate ?relation_document_date ;
            fp:documentQuote ?relation_document_quote .
  FILTER (?relation_as_of <= "2026-07-11"^^xsd:date)
  ?child rdfs:label ?child_name .
  ?security fp:issuedByCompany ?child .
  ?holding fp:holdingSecurity ?security ; fp:asOf ?holding_as_of ;
           fp:sourceId ?holding_source ; fp:supportedBy ?holding_document .
  ?holding_document a fp:Document ; fp:documentTitle ?holding_document_title ;
           fp:documentPublisher ?holding_document_publisher ;
           fp:documentPublishedDate ?holding_document_date ;
           fp:documentQuote ?holding_document_quote .
  FILTER (?holding_as_of <= "2026-07-11"^^xsd:date)
  OPTIONAL { ?holding fp:weight ?weight }
  ?etf a fp:ETF ; fp:hasHolding ?holding ; rdfs:label ?etf_name .
}
ORDER BY ?etf_name ?child_name
"""


def ecopro_subsidiary_etfs() -> list[dict]:
    return sparql(ECOPRO_HOLDING_QUERY)


def evidence_coverage() -> dict:
    """Store 전체의 운영 대상 Graph 관계 evidence 계약을 집계한다."""
    result = {}
    for label, cls in (("holding", "Holding"), ("subsidiary_relation", "SubsidiaryRelation")):
        total = sparql(f"""
PREFIX fp: <http://mafest.ai/product#>
SELECT (COUNT(DISTINCT ?relation) AS ?count) WHERE {{ ?relation a fp:{cls} . }}
""")[0]["count"]
        supported = sparql(f"""
PREFIX fp: <http://mafest.ai/product#>
SELECT (COUNT(DISTINCT ?relation) AS ?count) WHERE {{
  ?relation a fp:{cls} ; fp:supportedBy ?document .
  ?document a fp:Document .
}}
""")[0]["count"]
        total, supported = int(total), int(supported)
        result[label] = {"total": total, "supported": supported,
                         "coverage": supported / total if total else 1.0}
    return result



# === 실험 확장 (EXP-20260828-graph-01) — 채택 전까지 노트북에만 존재 ===
# 아래 템플릿의 어휘(predicate·URI 패턴)는 로컬 store(1,628,311 triples, cutoff 2026-08-24)
# 실측 프로브로 검증된 것이다:
#   상품: fpi:{etfkr|etfgl|fund|bond}-<코드>, fp:productShortName/productName/productCode
#   기업: fpi:corp-<코드>, fp:organizationName + rdfs:label + skos:altLabel
#   편입: 상품 -fp:hasHolding-> Holding{holdingSecurity, weight, asOf, sourceId, supportedBy}
#   자회사: 기업 -fp:hasSubsidiary-> SubsidiaryRelation{subsidiaryCompany, ownershipPct, asOf, supportedBy}
#   증권↔기업: ?sec fp:issuedByCompany ?corp / 채권 발행: ?bond fp:issuedBy ?issuer
PREFIXES = (
    "PREFIX fp: <http://mafest.ai/product#>\n"
    "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n"
    "PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n"
    "PREFIX skos: <http://www.w3.org/2004/02/skos/core#>\n"
    "PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>\n"
)
DATA_CUTOFF = "2026-08-24"


def _lit(text: str) -> str:
    return '"' + str(text).replace("\\", "\\\\").replace('"', '\\"') + '"'


def _q(body: str) -> list:
    return sparql(PREFIXES + body)


def _result(rows, status=None, **extra):
    out = {"status": status or ("ok" if rows else "empty"), "rows": rows}
    out.update(extra)
    return out


def product_info(short_name: str) -> dict:
    """G01/G02: 상품 URI·정식명·코드·투자지역."""
    rows = _q(f"""
SELECT ?product ?name ?code ?region_label WHERE {{
  ?product fp:productShortName {_lit(short_name)} .
  OPTIONAL {{ ?product fp:productName ?name }}
  OPTIONAL {{ ?product fp:productCode ?code }}
  OPTIONAL {{ ?product fp:hasInvestmentRegion ?r . ?r rdfs:label ?region_label .
             FILTER(lang(?region_label) = "ko") }}
}}""")
    return _result(rows)


def product_classifications(short_name: str) -> dict:
    """G08: 상품에 연결된 분류 개체(투자지역·자산유형·테마·위험등급 등)."""
    rows = _q(f"""
SELECT ?pred ?node ?node_label ?node_type WHERE {{
  ?product fp:productShortName {_lit(short_name)} ; ?pred ?node .
  ?node rdf:type ?node_type .
  FILTER(?node_type IN (fp:InvestmentRegion, fp:AssetType, fp:Theme, fp:RiskGrade,
                        fp:FundType, fp:Currency, fp:ManagementStrategy, fp:LeverageType))
  OPTIONAL {{ ?node rdfs:label ?node_label . FILTER(lang(?node_label) = "ko") }}
}}""")
    return _result(rows)


def company_info(name: str) -> dict:
    """G03: 기업 URI + 등록된 다른 이름(rdfs:label, skos:altLabel)."""
    rows = _q(f"""
SELECT ?company ?label ?alt WHERE {{
  ?company rdf:type fp:Company ; fp:organizationName {_lit(name)} .
  OPTIONAL {{ ?company rdfs:label ?label }}
  OPTIONAL {{ ?company skos:altLabel ?alt }}
}}""")
    return _result(rows)


def subsidiaries(company_name: str, limit: int = 30) -> dict:
    """G04: 자회사 관계 + 기준일(asOf)·출처(sourceId, supportedBy 문서 제목)."""
    rows = _q(f"""
SELECT ?relation ?child_name ?ownership_pct ?as_of ?source ?doc_title WHERE {{
  ?parent rdf:type fp:Company ; fp:organizationName {_lit(company_name)} ;
          fp:hasSubsidiary ?relation .
  ?relation fp:subsidiaryCompany ?child .
  ?child fp:organizationName ?child_name .
  OPTIONAL {{ ?relation fp:ownershipPct ?ownership_pct }}
  OPTIONAL {{ ?relation fp:asOf ?as_of }}
  OPTIONAL {{ ?relation fp:sourceId ?source }}
  OPTIONAL {{ ?relation fp:supportedBy ?doc . ?doc fp:documentTitle ?doc_title }}
  FILTER(!BOUND(?as_of) || ?as_of <= "{DATA_CUTOFF}"^^xsd:date)
}} ORDER BY ?child_name LIMIT {int(limit)}""")
    return _result(rows)


def product_holdings(short_name: str, limit: int = 10) -> dict:
    """G05: 상품 → 편입 증권 + 비중 + 기준일."""
    rows = _q(f"""
SELECT ?security_label ?weight ?as_of ?source WHERE {{
  ?product fp:productShortName {_lit(short_name)} ; fp:hasHolding ?h .
  ?h fp:holdingSecurity ?sec .
  OPTIONAL {{ ?sec rdfs:label ?security_label }}
  OPTIONAL {{ ?h fp:weight ?weight }}
  OPTIONAL {{ ?h fp:asOf ?as_of }}
  OPTIONAL {{ ?h fp:sourceId ?source }}
  FILTER(!BOUND(?as_of) || ?as_of <= "{DATA_CUTOFF}"^^xsd:date)
}} ORDER BY DESC(?weight) LIMIT {int(limit)}""")
    return _result(rows)


def etfs_holding_security(security_label: str, limit: int = 20) -> dict:
    """G06/G12: 증권 라벨 → 역방향 → 편입 ETF."""
    rows = _q(f"""
SELECT DISTINCT ?etf ?etf_name ?weight ?as_of WHERE {{
  ?sec rdf:type fp:Security ; rdfs:label {_lit(security_label)} .
  ?h fp:holdingSecurity ?sec .
  ?etf rdf:type fp:ETF ; fp:hasHolding ?h ; fp:productShortName ?etf_name .
  OPTIONAL {{ ?h fp:weight ?weight }}
  OPTIONAL {{ ?h fp:asOf ?as_of }}
}} ORDER BY DESC(?weight) LIMIT {int(limit)}""")
    return _result(rows)


def subsidiary_holding_etfs(company_name: str, limit: int = 50) -> dict:
    """G07: 기업 → 자회사 → (자회사 발행 증권) → 편입 ETF. (ecopro 하드코딩 일반화)"""
    rows = _q(f"""
SELECT DISTINCT ?etf_name ?child_name ?security_label ?weight ?holding_as_of WHERE {{
  ?parent rdf:type fp:Company ; fp:organizationName {_lit(company_name)} ;
          fp:hasSubsidiary ?relation .
  ?relation fp:subsidiaryCompany ?child .
  ?child fp:organizationName ?child_name .
  ?sec fp:issuedByCompany ?child .
  OPTIONAL {{ ?sec rdfs:label ?security_label }}
  ?h fp:holdingSecurity ?sec .
  OPTIONAL {{ ?h fp:weight ?weight }}
  OPTIONAL {{ ?h fp:asOf ?holding_as_of }}
  ?etf rdf:type fp:ETF ; fp:hasHolding ?h ; fp:productShortName ?etf_name .
  FILTER(!BOUND(?holding_as_of) || ?holding_as_of <= "{DATA_CUTOFF}"^^xsd:date)
}} ORDER BY ?etf_name LIMIT {int(limit)}""")
    return _result(rows)


def bond_info(product_name: str) -> dict:
    """G09: 채권 종류(rdf:type)·발행사·신용등급."""
    rows = _q(f"""
SELECT ?bond ?cls ?issuer_name ?rating WHERE {{
  ?bond fp:productName {_lit(product_name)} ; rdf:type ?cls .
  FILTER(?cls IN (fp:CorporateBond, fp:GovernmentBond, fp:SpecialBond, fp:Bond))
  OPTIONAL {{ ?bond fp:issuedBy ?issuer . ?issuer fp:organizationName ?issuer_name }}
  OPTIONAL {{ ?bond fp:hasCreditRating ?r . BIND(REPLACE(STR(?r), ".*#Rating_", "") AS ?rating) }}
}}""")
    return _result(rows)


def fund_holdings_check() -> dict:
    """G10: 펀드 편입 데이터 적재 여부 — empty(관계 없음)와 data_gap(미적재)을 구분."""
    funds = int(_q("SELECT (COUNT(DISTINCT ?f) AS ?n) WHERE { ?f rdf:type fp:PublicFund . ?f fp:hasHolding ?h }")[0]["n"])
    products = int(_q("SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { ?p fp:hasHolding ?h }")[0]["n"])
    total_funds = int(_q("SELECT (COUNT(DISTINCT ?f) AS ?n) WHERE { ?f rdf:type fp:PublicFund }")[0]["n"])
    status = "ok" if funds else ("data_gap" if products else "empty")
    note = (f"편입 관계 보유 상품 {products}개(전부 ETF), 펀드 {total_funds}개 중 0개 → "
            "펀드 편입 데이터 미적재(data_gap). '펀드가 주식을 편입하지 않는다'가 아니다."
            if status == "data_gap" else "")
    return {"status": status, "rows": [], "funds_with_holdings": funds,
            "products_with_holdings": products, "total_funds": total_funds, "note": note}


def domain_violation(short_name: str, prop: str = "issuedBy") -> dict:
    """G11: TBox rdfs:domain 과 주어 클래스 비교 — 잘못된 온톨로지 관계 거부."""
    subj = _q(f"SELECT DISTINCT ?product ?cls WHERE {{ ?product fp:productShortName {_lit(short_name)} ; rdf:type ?cls }}")
    if not subj:
        return {"status": "empty", "rows": [], "note": "주어 상품 부재"}
    domains = [d["d"] for d in _q(f"SELECT ?d WHERE {{ fp:{prop} rdfs:domain ?d }}")]
    ok_rows = []
    for s in subj:
        for dom in domains:
            if sparql(PREFIXES + f"ASK {{ <{s['cls']}> rdfs:subClassOf* <{dom}> }}"):
                ok_rows.append({"cls": s["cls"], "domain": dom})
    if ok_rows:
        return {"status": "ok", "rows": ok_rows}
    comment = _q(f"SELECT ?c WHERE {{ fp:{prop} rdfs:comment ?c }}")
    return {"status": "abstain_domain_error", "rows": [],
            "subject_classes": sorted({s["cls"] for s in subj}),
            "required_domain": domains,
            "tbox_comment": (comment[0]["c"][:300] if comment else "")}


def class_counts() -> dict:
    """체크리스트 3번: 주식·ETF·펀드·채권·기업이 모두 조회되는가."""
    out = {}
    for cls in ("Security", "ETF", "PublicFund", "CorporateBond",
                "GovernmentBond", "SpecialBond", "Company"):
        out[cls] = int(_q(f"SELECT (COUNT(?s) AS ?n) WHERE {{ ?s rdf:type fp:{cls} }}")[0]["n"])
    return out


registered: tools.graph


# Graph 테스트 질문 G01~G12 (계획 v4 §5)

In [8]:
GRAPH_TEST_QUESTIONS = [
    {"id": "G01", "question": "KODEX 200의 상품 URI, 정식 상품명, 상품 코드, 투자지역을 알려줘", "expected": "success"},
    {"id": "G02", "question": "해외 ETF VOO의 상품명, 상품 코드, 투자지역을 알려줘", "expected": "success"},
    {"id": "G03", "question": "삼성전자라는 기업의 Graph URI와 등록된 다른 이름을 알려줘", "expected": "success"},
    {"id": "G04", "question": "에코프로와 연결된 자회사 관계를 알려줘. 관계 기준일과 출처도 함께 보여줘", "expected": "success_or_partial"},
    {"id": "G05", "question": "KODEX 200이 편입한 증권 10개와 각각의 편입 비중을 알려줘", "expected": "success"},
    {"id": "G06", "question": "삼성전자를 편입한 국내 ETF를 알려줘", "expected": "success_or_partial"},
    {"id": "G07", "question": "에코프로의 자회사를 편입한 국내 ETF를 찾아줘", "expected": "success_or_partial"},
    {"id": "G08", "question": "KODEX 200에 연결된 투자지역과 자산유형 분류를 알려줘", "expected": "success"},
    {"id": "G09", "question": "현대해상화재보험7(후)(콜/후)의 채권 종류와 발행사를 알려줘", "expected": "success_or_partial"},
    {"id": "G10", "question": "공모펀드가 편입한 개별 주식과 편입 비중을 알려줘", "expected": "data_gap"},
    {"id": "G11", "question": "VOO가 직접 발행한 회사채를 찾아줘", "expected": "abstain_domain_error"},
    {"id": "G12", "question": "존재하지 않는 가상의 ETF가 편입한 종목을 알려줘", "expected": "empty"},
]

In [9]:
import time

from tools import graph as G

CALLS = {
    "G01": lambda: G.product_info("KODEX 200"),
    "G02": lambda: G.product_info("VOO"),
    "G03": lambda: G.company_info("삼성전자"),
    "G04": lambda: G.subsidiaries("에코프로"),
    "G05": lambda: G.product_holdings("KODEX 200", 10),
    "G06": lambda: G.etfs_holding_security("삼성전자"),
    "G07": lambda: G.subsidiary_holding_etfs("에코프로"),
    "G08": lambda: G.product_classifications("KODEX 200"),
    "G09": lambda: G.bond_info("현대해상화재보험7(후)(콜/후)"),
    "G10": lambda: G.fund_holdings_check(),
    "G11": lambda: G.domain_violation("VOO", "issuedBy"),
    "G12": lambda: G.etfs_holding_security("존재하지 않는 가상의 ETF 편입 종목"),
}

# expected → 허용 status (success_or_partial 도 관계가 조회되면 ok 로 본다)
ACCEPT = {"success": {"ok"}, "success_or_partial": {"ok"},
          "data_gap": {"data_gap"}, "abstain_domain_error": {"abstain_domain_error"},
          "empty": {"empty"}}

GRAPH_RESULTS = []
for item in GRAPH_TEST_QUESTIONS:
    t0 = time.perf_counter()
    try:
        res = CALLS[item["id"]]()
    except Exception as exc:
        res = {"status": "error", "rows": [], "note": f"{type(exc).__name__}: {exc}"}
    elapsed = round((time.perf_counter() - t0) * 1000, 1)
    verdict = "PASS" if res["status"] in ACCEPT[item["expected"]] else "FAIL"
    GRAPH_RESULTS.append({"item": item, "res": res, "elapsed_ms": elapsed, "verdict": verdict})
    head = res["rows"][0] if res.get("rows") else res.get("note", "")
    print(f"[{item['id']}] {verdict}  status={res['status']}  rows={len(res.get('rows', []))}"
          f"  {elapsed}ms")
    print("   ", str(head)[:160])
    print("-" * 80)

[G01] PASS  status=ok  rows=1  143.5ms
    {'product': 'http://mafest.ai/instance/etfkr-KR7069500007', 'name': '삼성 KODEX200 증권상장지수투자신탁[주식]', 'code': 'KR7069500007', 'region_label': '국내'}
--------------------------------------------------------------------------------
[G02] PASS  status=ok  rows=1  1.0ms
    {'product': 'http://mafest.ai/instance/etfgl-VOO', 'name': 'Vanguard 500 Index Fund;ETF', 'code': 'VOO', 'region_label': '미국'}
--------------------------------------------------------------------------------
[G03] PASS  status=ok  rows=5  922.5ms
    {'company': 'http://mafest.ai/instance/corp-00126380', 'label': '삼성전자', 'alt': '삼성전자㈜'}
--------------------------------------------------------------------------------
[G04] PASS  status=ok  rows=22  807.4ms
    {'relation': 'http://mafest.ai/instance/sub-00536541-ECOPROAMERICAINC%2E-0', 'child_name': 'ECOPROAMERICAINC.', 'ownership_pct': '100', 'as_of': '2026-03-18', '
------------------------------------------------------------------

## 체크리스트 (계획 v4 §5)

`G10` 결과가 빈 배열이라고 해서 "공모펀드가 주식을 편입하지 않는다"고 판단하지 않는다 —
편입 데이터 미적재(data_gap)인지 먼저 구분한다.

In [10]:
# 계획 v4 §5 체크리스트 10항 자동 평가
S = {r["item"]["id"]: r["res"] for r in GRAPH_RESULTS}
counts = G.class_counts()

def rows(gid):
    return S[gid].get("rows") or []

checklist = [
    ("1. Store가 열리는가", man["triple_count"] > 0),
    ("2. Triple을 조회할 수 있는가", bool(rows("G01"))),
    ("3. 주식·ETF·펀드·채권·기업 모두 조회", all(v > 0 for v in counts.values())),
    ("4. 상품→편입 증권 이동", S["G05"]["status"] == "ok"),
    ("5. 기업→자회사 이동", S["G04"]["status"] == "ok"),
    ("6. 증권→ETF 역방향 이동", S["G06"]["status"] == "ok"),
    ("7. as_of가 관계 데이터에 포함", any(r.get("as_of") for r in rows("G04"))
        and any(r.get("as_of") for r in rows("G05"))),
    ("8. supportedBy 문서 근거 포함", any(r.get("doc_title") for r in rows("G04"))),
    ("9. empty와 data_gap 구분", S["G10"]["status"] == "data_gap" and S["G12"]["status"] == "empty"),
    ("10. 잘못된 온톨로지 관계 거부", S["G11"]["status"] == "abstain_domain_error"),
]
print("class counts:", counts)
for name, ok in checklist:
    print(("PASS " if ok else "FAIL "), name)

class counts: {'Security': 11879, 'ETF': 7207, 'PublicFund': 14716, 'CorporateBond': 12622, 'GovernmentBond': 1775, 'SpecialBond': 6100, 'Company': 26167}
PASS  1. Store가 열리는가
PASS  2. Triple을 조회할 수 있는가
PASS  3. 주식·ETF·펀드·채권·기업 모두 조회
PASS  4. 상품→편입 증권 이동
PASS  5. 기업→자회사 이동
PASS  6. 증권→ETF 역방향 이동
PASS  7. as_of가 관계 데이터에 포함
PASS  8. supportedBy 문서 근거 포함
PASS  9. empty와 data_gap 구분
PASS  10. 잘못된 온톨로지 관계 거부


In [11]:
records = []
for r in GRAPH_RESULTS:
    res = dict(r["res"])
    res["rows"] = res.get("rows", [])[:10]
    records.append({"experiment_id": "EXP-20260828-graph-01",
                    "question_id": r["item"]["id"],
                    "question": r["item"]["question"],
                    "expected": r["item"]["expected"],
                    "verdict": r["verdict"],
                    "elapsed_ms": r["elapsed_ms"],
                    **res})
out_path = RESULTS_DIR / "graph_0828.json"
out_path.write_text(json.dumps({"records": records,
                                "checklist": [{"item": n, "ok": ok} for n, ok in checklist],
                                "class_counts": counts},
                               ensure_ascii=False, indent=2, default=str),
                    encoding="utf-8")
print("saved:", out_path)

saved: c:\Users\rladl\Desktop\2026_MIRAE_ASSET_AI-Festival\2026_10th_MIRAE-ASSET_AI-Festival\test\notebook\experiments\results\graph_0828.json
